# 02. XGBoost Modeling

이 노트북은 10분/15분 데이터 각각에 대해 XGBoost 모델을 학습하고, Accuracy/F1/ROC-AUC와 피처 중요도를 확인합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.train_xgboost import train_xgboost

DATASETS = [
    (PROJECT_ROOT / "data" / "Challenger_Ranked_Games_10minute.csv", "10minute"),
    (PROJECT_ROOT / "data" / "Challenger_Ranked_Games_15minute.csv", "15minute"),
]

## 빠른 실행

처음 실험할 때는 `quick=True`로 전체 코드가 정상 작동하는지 확인합니다.
최종 결과를 만들 때는 `quick=False`로 바꿔서 실행하면 됩니다.

In [ ]:
results = []
for data_file, label in DATASETS:
    metrics = train_xgboost(
        data_file=data_file,
        time_label=label,
        output_dir=PROJECT_ROOT / "results",
        model_dir=PROJECT_ROOT / "models",
        tune=True,
        quick=True,   # 최종 실험에서는 False 권장
    )
    results.append(metrics)

pd.DataFrame(results)

## 저장된 결과 확인

생성되는 주요 파일:
- `results/tables/xgboost_10minute_metrics.csv`
- `results/tables/xgboost_15minute_metrics.csv`
- `results/tables/xgboost_10minute_feature_importance.csv`
- `results/figures/xgboost_10minute_feature_importance.png`
- `results/figures/xgboost_10minute_confusion_matrix.png`
- `results/figures/xgboost_10minute_roc_curve.png`

In [ ]:
comparison = pd.concat([
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_10minute_metrics.csv"),
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_15minute_metrics.csv"),
], ignore_index=True)
comparison[["model", "time_label", "accuracy", "precision", "recall", "f1", "roc_auc"]]

In [ ]:
importance10 = pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_10minute_feature_importance.csv")
importance15 = pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_15minute_feature_importance.csv")

importance10.head(15), importance15.head(15)

## 발표용 해석 방향

XGBoost는 tabular data에서 강한 기본 성능을 보이는 모델입니다. 특히 피처 중요도를 통해 `골드 차이`, `레벨 차이`, `킬/어시스트 차이`, `오브젝트 차이` 중 어떤 요소가 승패 예측에 크게 작용했는지 설명할 수 있습니다.

<!-- UPDATED_XGBOOST_IMPROVED -->
## 추가 실험: Threshold Tuning과 개선된 XGBoost 설정

기본 XGBoost 실험은 0.5 threshold를 사용합니다. 추가 실험에서는 train split 안에서 validation split을 한 번 더 만들고, validation 예측 확률을 기준으로 decision threshold를 탐색합니다.

또한 `gamma`, `reg_alpha`, 더 넓은 `reg_lambda`, 더 큰 `n_estimators` 후보를 추가하여 XGBoost 탐색 범위를 넓혔습니다. 이 실험은 기존 baseline 파일을 덮어쓰지 않고 `threshold` suffix를 붙여 저장합니다.

중요한 점은 threshold를 test set에서 맞추면 데이터 누수가 된다는 것입니다. 이 프로젝트에서는 validation 기준으로만 threshold를 선택하고, test set은 최종 평가에만 사용합니다.

In [ ]:
# 시간이 부족하면 quick=True로 동작 확인만 하고,
# 최종 제출 결과를 다시 만들 때는 quick=False로 실행하는 것을 권장합니다.
threshold_results = []
for data_file, label in DATASETS:
    metrics = train_xgboost(
        data_file=data_file,
        time_label=label,
        output_dir=PROJECT_ROOT / "results",
        model_dir=PROJECT_ROOT / "models",
        tune=True,
        quick=True,
        tune_threshold=True,
        threshold_metric="accuracy",
        experiment_label="threshold",
    )
    threshold_results.append(metrics)

pd.DataFrame(threshold_results)[[
    "model", "experiment_label", "time_label", "accuracy", "precision", "recall",
    "f1", "roc_auc", "decision_threshold", "threshold_metric"
]]

In [ ]:
# Baseline과 threshold XGBoost 결과 비교
xgb_compare = pd.concat([
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_10minute_metrics.csv"),
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_10minute_threshold_metrics.csv"),
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_15minute_metrics.csv"),
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_15minute_threshold_metrics.csv"),
], ignore_index=True, sort=False)

xgb_compare["experiment_label"] = xgb_compare["experiment_label"].fillna("baseline").replace("", "baseline")
xgb_compare[[
    "model", "experiment_label", "time_label", "accuracy", "precision", "recall", "f1", "roc_auc",
    "decision_threshold"
]]

## XGBoost 해석 포인트

XGBoost는 tabular data에서 성능이 안정적이고, feature importance를 통해 예측 근거를 설명할 수 있다는 장점이 있습니다. 본 프로젝트에서는 15분 XGBoost baseline이 단일 모델 중 가장 안정적인 대표 모델로 볼 수 있습니다.

추가 threshold 실험은 Accuracy를 무조건 크게 올리기 위한 장치라기보다, Precision과 Recall의 균형을 조정해 모델의 의사결정 기준을 분석하는 과정입니다. 따라서 보고서에서는 baseline XGBoost를 메인 모델로 두고, threshold tuning은 추가 개선 실험으로 소개하는 것이 적절합니다.